In [1]:
# 다중분류 / 뉴스 카테고리
import urllib
url = "https://raw.githubusercontent.com/kocohub/korean-hate-speech/master/labeled/train.tsv"
def download_data():
    urllib.request.urlretrieve(url, 'train_hate_tsv')

download_data()

In [2]:
import pandas as pd
df = pd.read_csv('train_hate_tsv', sep='\t').dropna()
# None 없음 hate 싫음 offensive 공격적
df.head()

,comments,contain_gender_bias,bias,hate
0,(현재 호텔주인 심정) 아18 난 마른하늘에 날벼락맞고 호텔망하게생겼는데 누군 계속...,False,others,hate
1,....한국적인 미인의 대표적인 분...너무나 곱고아름다운모습...그모습뒤의 슬픔을...,False,none,none
2,"...못된 넘들...남의 고통을 즐겼던 넘들..이젠 마땅한 처벌을 받아야지..,그래...",False,none,hate
3,"1,2화 어설펐는데 3,4화 지나서부터는 갈수록 너무 재밌던데",False,none,none
4,1. 사람 얼굴 손톱으로 긁은것은 인격살해이고2. 동영상이 몰카냐? 메걸리안들 생각...,True,gender,hate


In [3]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['hate'])
label_encoder.classes_

array(['hate', 'none', 'offensive'], dtype=object)

In [4]:
#학습/테스트데이터 분리(8:2)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df,test_size=0.2,random_state=42)
train_df.shape, test_df.shape, df.shape

((6316, 5), (1580, 5), (7896, 5))

In [25]:
# 전처리 및 어휘사전(vocab) 구축
from konlpy.tag import Okt
okt = Okt()
# okt.pos('아버지가 방에 들어가신다'), okt.morphs('아버지가 방에 들어가신다'), okt.phrases('아버지가 방에 들어가신다')
def konlpy_tokenizer(text):
    return okt.morphs(text)

train_tokenized = []
for text in train_df['comments']:
    tokens = konlpy_tokenizer(text)
    train_tokenized.append(tokens)

test_tokenized = [ konlpy_tokenizer(text) for text in test_df['comments']]

In [26]:
from collections import Counter
import torch

#빈도수 기준 단어사전 생성
vocab_size = 8000
all_tokens = [ token for token_list in train_tokenized  for token in token_list  ]
word_counts = Counter(all_tokens)
vocab = {'<PAD>':0, '<UNK>':1}

for word in set([word for word,cnt in word_counts.most_common(vocab_size-2)]):
    vocab[word] = len(vocab)

# 문장의 길이를 통일  시퀀스
import torch
sequence_len = 30
def text_to_sequence(tokens, max_len=30):
    seq = [vocab.get(token,1) for token in tokens]        
    if len(seq) < max_len:
        seq += [0]*(max_len - len(seq))
    else:
        seq = seq[ : max_len]
    return seq
# tensor 변환
x_train = torch.LongTensor([text_to_sequence(tokens) for tokens in train_tokenized])
y_train = torch.LongTensor(train_df['labels'].values)

x_test = torch.LongTensor([text_to_sequence(tokens) for tokens in test_tokenized])
y_test = torch.LongTensor(test_df['labels'].values)

# 데이터로더
batch_size = 64
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader( TensorDataset(x_train,y_train), batch_size=batch_size,shuffle=True )
test_loader = DataLoader( TensorDataset(x_test, y_test),batch_size=batch_size )

In [27]:
# 모델정의
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class TextClassifier(nn.Module):
    def __init__(self, model_type, vocab_size, embed_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.model_type = model_type
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        if model_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'CNN':
            self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=3)
            # 이미지 batch, channel, height, width
            # conv1d  batch, channel, length
            # 텍스트임베딩은 단어순서가 보통 2번째에 위치--> 채널위치하고 바꿔주는 작업
            # kernel_size=3 은 3-grame

            
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        embedded = self.embedding(x)  # [64, 30, 128]  [Batch, seq_len ,embeded_dim]    
        if self.model_type =='CNN':
            embedded = embedded.transpose(1,2)
            conveled = torch.relu(self.conv(embedded))
            hidden_out = torch.max(conveled, dim=2)[0]
        elif self.model_type == 'LSTM':
            _,(hidden,_) = self.rnn(embedded)
            hidden_out = hidden[-1]        
        else: # RNN OR GRU
            output,hidden = self.rnn(embedded) # torch.Size([64, 30, 256]) torch.Size([1, 64, 256])       
            hidden_out = hidden[-1]
        return self.fc(hidden_out)  # 64,256

In [30]:
# 학습 및 평가용
def train_and_evaluation(model_type ='RNN', epochs=10):
    embed_dim = 128
    hidden_dim = 256
    output_dim = len(y_train.unique())
    tc = TextClassifier(model_type, vocab_size, embed_dim, hidden_dim, output_dim).to(device)
    criterion = nn.CrossEntropyLoss() # softmax 내장
    optimizer = torch.optim.Adam(tc.parameters())
    for epoch in range(epochs):
        tc.train()
        total_loss = 0
        for b_x, b_y in train_loader:
            optimizer.zero_grad()
            b_x, b_y = b_x.to(device), b_y.to(device)
            output = tc(b_x)  # forward
            loss = criterion(output,b_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        # 검증
        tc.eval()
        correct = 0
        with torch.no_grad():
            for b_x, b_y in test_loader:
                b_x, b_y = b_x.to(device), b_y.to(device)
                outputs = tc(b_x)
                _,pred = torch.max(outputs,1)
                correct += (pred == b_y).sum().item()
        acc = correct / len(test_df)
        print(f'epoch {epoch+1} / {epochs}  loss : {total_loss / len(train_loader):.4f}  acc : {acc:.4f}')

In [ ]:
train_and_evaluation('RNN')  # RNN LSTM GRU CNN

KeyboardInterrupt: 